In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [8]:
import json
from pydantic import ValidationError
import cv2

In [3]:
from src.schemas.coco_dataset import COCODataset

In [4]:
dataset_dir = Path("datasets/minecraft")

In [5]:
def check_annotation_structure(path: Path) -> bool:
    with open(path, mode="r") as file:
        data = json.load(file)
        try:
            COCODataset(**data)
            return True
        except Exception as e:
            raise e 
        

In [ ]:

def check_annotations_and_images(data_dir: Path) -> bool:
    image_extensions = ['jpg', 'jpeg', 'png', 'webp', 'gif', 'svg', 'bmp', 'tiff', 'heic', 'avif']
    annotations = data_dir / "annotations.json"
    if not annotations.is_file():
        raise FileNotFoundError("Файл аннтоаций не найден")

    images = set([f for f in os.listdir(data_dir) if f.endswith(image_extensions)])
    for image in images:
        try:
            cv2.imread(image)
        except Exception as e:
            images.remove(image)
        
    with open(annotations, mode="r") as f:
        data = json.load(f)
    
    annotated_images = set([d["images"]["file_name"] for d in data])
    
    if len(annotated_images) != len(images) or len(annotated_images - images) != 0:
        print("Images - Annotations: ", images - annotated_images)
        print("Annotations - Images: ", annotated_images - images)
    else:
        print("Ok")
        
    
    
    os

In [ ]:
dirs = ["train", "valid", "test"]

for dir in dirs:
    print(f"Проверка аннотаций в {dir}")
    try:
        checked = check_annotation_structure(dataset_dir / dir / "annotations.json")
    except ValidationError as e:
        print("❌ ОШИБКА ВАЛИДАЦИИ СТРУКТУРЫ COCO!")
        print(f"Всего обнаружено нестыковок: {e.error_count()}\n")
        print("--- ТОП-5 ПРОБЛЕМНЫХ МЕСТ В ДАННЫХ ---")
        
        for i, error in enumerate(e.errors()[:5]):
            path_to_error = " -> ".join(str(loc) for loc in error['loc'])
            error_type = error['type']
            error_msg = error['msg']
        
            bad_value = error.get('input', 'Не удалось определить')
            
            print(f"Ошибка #{i+1}:")
            print(f"  📍 Где искать: {path_to_error}")
            print(f"  📝 Что не так: {error_msg} (тип ошибки: {error_type})")
            print(f"  ❌ Что там записано сейчас: {bad_value}")
            print("-" * 40)
    print(f"Проверено {dir}")
print("Все ок!")

Проверка аннотаций в train
Проверено train
Проверка аннотаций в valid
Проверено valid
Проверка аннотаций в test
Проверено test
Все ок!
